# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [59]:
#Libraries
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix, classification_report, recall_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier

from xgboost import XGBClassifier

In [60]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [61]:
#Check Null values
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [62]:
# Drop the Null values
spaceship = spaceship.dropna()
spaceship.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [63]:
# Extract deck (first letter of Cabin)
spaceship["Cabin"] = spaceship["Cabin"].str[0]
spaceship.head(3)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False


In [64]:
spaceship_clean = spaceship.drop(columns=["PassengerId", "Name"], errors="ignore")
spaceship_clean.head(3)

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False


In [77]:
# Check the Transported (1) vs Not Transported (0)
((spaceship_clean['Transported'].value_counts() / len(spaceship_clean)).round(4) * 100)

Transported
True     50.36
False    49.64
Name: count, dtype: float64

In [65]:
# For non-numerical columns, do dummies.
spaceship_clean = pd.get_dummies(spaceship_clean, drop_first=True)
spaceship_clean.head(3)

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [66]:
features = spaceship_clean.drop(columns = ["Transported"])
target = spaceship_clean["Transported"]

**Perform Train Test Split**

In [67]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size = 0.20, random_state=0)

In [68]:
scaler = StandardScaler()

In [69]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [70]:
X_train_scaled

array([[ 0.22051462, -0.34704629, -0.28209937, ..., -0.32826333,
         0.67709663, -0.16314854],
       [-1.70452544, -0.34704629, -0.28209937, ..., -0.32826333,
         0.67709663, -0.16314854],
       [ 0.08301176, -0.34704629, -0.14334471, ..., -0.32826333,
         0.67709663, -0.16314854],
       ...,
       [ 0.49552034, -0.34704629,  2.51415978, ..., -0.32826333,
         0.67709663, -0.16314854],
       [ 0.35801748, -0.34704629, -0.2797476 , ..., -0.32826333,
         0.67709663,  6.12938358],
       [-1.01701113, -0.34393854,  0.23587883, ..., -0.32826333,
         0.67709663, -0.16314854]], shape=(5284, 19))

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

In [80]:
datasets = [
    {"name": "raw",        "X_train": X_train,      "X_test": X_test},
    {"name": "scaled",        "X_train": X_train_scaled,      "X_test": X_test_scaled}
]

In [75]:
models = [
    {"name": "Bagging",          "model": BaggingClassifier(DecisionTreeClassifier(max_depth=20), n_estimators=100, max_samples=1000)},
    {"name": "RandomForest",     "model": RandomForestClassifier(n_estimators=100, max_depth=20)},
    {"name": "AdaBoost",         "model": AdaBoostClassifier(DecisionTreeClassifier(max_depth=20), n_estimators=100)},
    {"name": "GradientBoosting", "model": GradientBoostingClassifier(n_estimators=100, max_depth=20)},
    {"name": "XGBoost",          "model": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1)}
]

In [81]:
results = []

for ds in datasets:
    for m in models:
        m["model"].fit(ds["X_train"], y_train)
        pred = m["model"].predict(ds["X_test"])

        results.append({
            "dataset": ds["name"],
            "model":   m["name"],
            "F1Score":    f1_score(y_test, pred),
        })
        print(f"{ds['name']:>10} | {m['name']:<16} done")

results_df = pd.DataFrame(results).sort_values("F1Score", ascending=False).reset_index(drop=True)
results_df

       raw | Bagging          done
       raw | RandomForest     done
       raw | AdaBoost         done
       raw | GradientBoosting done
       raw | XGBoost          done
    scaled | Bagging          done
    scaled | RandomForest     done
    scaled | AdaBoost         done
    scaled | GradientBoosting done
    scaled | XGBoost          done


,dataset,model,F1Score
0,raw,XGBoost,0.790560
1,scaled,XGBoost,0.790560
2,raw,RandomForest,0.790310
3,raw,GradientBoosting,0.790215
4,scaled,GradientBoosting,0.787834
5,raw,AdaBoost,0.779610
6,scaled,Bagging,0.779043
7,scaled,RandomForest,0.778287
8,raw,Bagging,0.774637
9,scaled,AdaBoost,0.770849


Which model is the best and why?

In [ ]:
# Random Forest and XGBoost perform best
# - Tree-based models excel at capturing interactions automatically
# - The dataset contains many categorical and numerical values >> Tree-based models: handle mixed features very well (after encoding) and do NOT assume linear relationships
# - Tree models: are robust to outliers, handle imperfect splits well, do not require scaling